In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V10  —  3-Day Safety Floor System
#  Terminal Logic (revised):
#   RUNNER   : ALL required terminals must have >= 85% of daily indent qty
#              AND enough stock for at least MIN_RUN_HOURS production
#   REPEATER : terminals must cover FULL daily indent (not just min run hrs)
#   STRANGER : terminals must cover FULL daily indent (not just min run hrs)
#              If terminals only cover partial < daily, part is BLOCKED
#   Planned qty is always capped to what terminals can actually supply
#
#  Fixed Machine Logic (revised):
#   - VT_Fixed sheet maps machines → their dedicated parts
#   - If fixed part inventory < 3 days → machine is RESERVED for that fixed part ONLY
#   - If ALL fixed parts on a machine have inventory >= 3 days → machine is OPEN
#     for any compatible part (Runner > Repeater > Stranger)
#   - Fixed parts ALWAYS prefer their fixed machine first
#
#  Priority (enforced everywhere): Runner >>>> Repeater >>>> Stranger
# =============================================================

PLANNING_DATE = date(2026, 4, 8)
INDENT_MONTH  = date(2026, 4,  1)

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

SAFETY_DAYS  = 3          # single ceiling — 3-day target

OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 3.0
OPD_SCENARIO_3 = 3.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

STRANGER_BASE_MULTIPLIER = 3.0
STRANGER_RATE_ANCHOR     = 100.0
# Terminal thresholds
RUNNER_TERMINAL_MIN_PCT  = 85.0   # Runner: all terminals must have >= 85% of daily indent
TERMINAL_HIGHLY_CRITICAL_PCT = 75.0
TERMINAL_CRITICAL_PCT        = 80.0

UTIL_TARGET_PCT = 100.0
RUNNER_DEFER_MACHINE_UTIL_PCT = 95
book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V10_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(1 for d in range(1, total + 1) if date(year, month, d).weekday() == 6)
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V10  —  3-Day Safety Floor System")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Safety ceiling: {SAFETY_DAYS} days  (3-day system)")
print(f"  OPD cap       : max {OPD_SCENARIO_3}× daily")
print(f"  Utilisation   : {UTIL_TARGET_PCT:.0f}% ({AVAILABLE_HOURS}h)")
print(f"  Runner tools  : MINIMUM needed to cover daily indent")
print(f"  Repeater/Stranger : SINGLE TOOL only")
print(f"  Priority order: Runner >>>> Repeater >>>> Stranger (ALWAYS enforced)")
print(f"  Terminal gate : ACTIVE")
print(f"    Runner       : ALL terminals >= {RUNNER_TERMINAL_MIN_PCT}% daily indent + min run hrs")
print(f"    Rep/Stranger : terminals must cover FULL daily indent")
print(f"  Fixed machines: Reserved when fixed part inv < {SAFETY_DAYS}d; open when >= {SAFETY_DAYS}d")
print(f"{'='*65}\n")

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")
vt_fixed_machines_raw = pd.read_excel(matrix_path,    sheet_name="VT_Fixed")

TERMINAL_PARTS_SHEET = "Part_Terminals"
TERMINAL_STOCK_SHEET = "Unavailable_Terminals"

try:
    _terminal_parts_raw   = pd.read_excel(terminal_path, sheet_name=TERMINAL_PARTS_SHEET)
    _terminal_stock_raw   = pd.read_excel(terminal_path, sheet_name=TERMINAL_STOCK_SHEET)
    _terminal_data_loaded = True
    print(f"  Terminal data loaded  ✓  "
          f"({len(_terminal_parts_raw)} part rows, "
          f"{len(_terminal_stock_raw)} terminal stock records)")
except FileNotFoundError:
    _terminal_data_loaded = False
    _terminal_parts_raw   = pd.DataFrame()
    _terminal_stock_raw   = pd.DataFrame()
    print(f"  WARNING: terminal_data.xlsx not found — Terminal gate DISABLED.")
except Exception as _e:
    _terminal_data_loaded = False
    _terminal_parts_raw   = pd.DataFrame()
    _terminal_stock_raw   = pd.DataFrame()
    print(f"  WARNING: terminal data load failed ({_e}) — Terminal gate DISABLED.")

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'.\nAvailable: {list(df.columns)}")
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(pts)} part(s)")

indent_daily = {p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# FIXED MACHINE ASSIGNMENTS
# =============================================================
# Rule: fixed part inv < 3-day safety → machine RESERVED for fixed part ONLY
# Rule: ALL fixed parts inv >= 3-day safety → machine OPEN for any compatible part
# Fixed parts ALWAYS plan on their fixed machine first (preference)
# Planning order on any machine: Runner > Repeater > Stranger
# =============================================================

def build_fixed_machine_map(df):
    """
    Returns:
      fixed_map  : { machine -> [fixed_parts] }
      part_fixed : { part -> [fixed_machines] }  (reverse map)
    """
    fixed_map  = {}
    part_fixed = {}
    if df is None or df.empty:
        return fixed_map, part_fixed
    cols = list(df.columns)
    if not cols:
        return fixed_map, part_fixed
    machine_col = cols[0]
    part_cols   = cols[1:]
    for _, row in df.iterrows():
        machine = str(row[machine_col]).strip()
        if not machine or machine.lower() in ("nan", ""):
            continue
        parts = []
        for pc in part_cols:
            val = row[pc]
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                parts.append(str(val).strip())
        if parts:
            fixed_map[machine] = parts
            for p in parts:
                part_fixed.setdefault(p, []).append(machine)
    return fixed_map, part_fixed

vt_fixed_map, vt_part_fixed_machines = build_fixed_machine_map(vt_fixed_machines_raw)

if vt_fixed_map:
    print(f"  Fixed machine assignments : {len(vt_fixed_map)} machines")
    for m, pts in sorted(vt_fixed_map.items()):
        print(f"    {m:<25} → {', '.join(pts)}")
else:
    print(f"  Fixed machine assignments : none")


def is_fixed_machine(machine):
    return machine in vt_fixed_map


def fixed_machine_is_open(machine, current_inventory):
    """
    Returns True if machine is OPEN for non-fixed parts.
    Open = ALL fixed parts on this machine have inventory >= SAFETY_DAYS * daily.
    Reserved (returns False) = ANY fixed part has inventory < SAFETY_DAYS * daily.
    """
    for fp in vt_fixed_map.get(machine, []):
        daily = indent_daily.get(fp, 0.0)
        inv   = current_inventory.get(fp, 0.0)
        if daily > 0 and inv < SAFETY_DAYS * daily:
            return False   # at least one fixed part below 3-day floor → RESERVED
    return True            # all fixed parts at/above ceiling → OPEN


def get_fixed_machines_for_part(part):
    """Returns list of machines that are fixed for this part."""
    return vt_part_fixed_machines.get(part, [])


# Global priority scores — populated during schedule()
priority_scores_global = {}

CATEGORY_TIER  = {"Runner": 0, "Repeater": 1, "Stranger": 2}
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# COLOUR CONSTRAINT
# =============================================================
COLOR_PURGE_HRS = 10 / 60.0

# =============================================================
# TERMINAL CONSTRAINT  (completely rewritten)
# =============================================================
# Rules:
#   RUNNER   : ALL required terminals must have stock >= 85% of daily_indent
#              AND stock must be enough for at least MIN_RUN_HOURS production
#              → if any terminal fails either check → part BLOCKED
#   REPEATER : terminals must supply enough to meet FULL daily indent
#   STRANGER : terminals must supply enough to meet FULL daily indent
#              → if max producible < daily_indent → part BLOCKED
#              → planned qty is capped to min(daily_indent, terminal_available_qty)
#
#   terminal_available_qty_for_part(part) → max qty this part can produce today
#     given current terminal running stock (= min stock across all required terminals)
# =============================================================

def _build_terminal_structures(parts_df, stock_df):
    part_terminals = {}
    if parts_df is None or (hasattr(parts_df, 'empty') and parts_df.empty):
        return part_terminals, {}, set(), {}
    cols      = list(parts_df.columns)
    part_col  = cols[0]
    term_cols = cols[1:]
    for _, row in parts_df.iterrows():
        part = str(row[part_col]).strip()
        if not part or part.lower() in ("nan", ""):
            continue
        required = []
        for tc in term_cols:
            val = row[tc]
            if pd.notna(val):
                val_str = str(val).strip()
                if val_str and val_str.lower() not in ("", "nan", "0", "0.0", "none", "-"):
                    required.append(val_str.upper())
        part_terminals[part] = required

    terminal_stock = {}
    reason_map     = {}
    if stock_df is not None and not (hasattr(stock_df, 'empty') and stock_df.empty):
        scols = list(stock_df.columns)
        def _find_col(candidates):
            for n in candidates:
                for c in scols:
                    if str(c).strip().lower() == n.lower():
                        return str(c).strip()
            return None
        tid_col    = _find_col(["Terminal_ID", "Terminal", "terminal_id", "ID"])
        stock_col  = _find_col(["Current_Stock", "Stock", "Qty", "Quantity", "current_stock"])
        reason_col = _find_col(["Reason", "reason", "Note", "Remark"])
        if tid_col and stock_col:
            for _, row in stock_df.iterrows():
                tid = str(row[tid_col]).strip().upper()
                if not tid or tid.lower() in ("nan", ""):
                    continue
                try:
                    stk = float(row[stock_col]) if pd.notna(row[stock_col]) else 0.0
                except (TypeError, ValueError):
                    stk = 0.0
                terminal_stock[tid] = stk
                if reason_col:
                    rv = str(row.get(reason_col, ""))
                    reason_map[tid] = "" if rv.lower() == "nan" else rv
        else:
            for _, row in stock_df.iterrows():
                vals = [v for v in row.values if pd.notna(v)]
                if len(vals) >= 2:
                    tid = str(vals[0]).strip().upper()
                    try:
                        stk = float(vals[1])
                    except (TypeError, ValueError):
                        stk = 0.0
                    if tid:
                        terminal_stock[tid] = stk

    unavailable_set = {tid for tid, stk in terminal_stock.items() if stk <= 0}
    terminal_detail = {
        tid: {"opening_stock": terminal_stock.get(tid, 0.0), "reason": reason_map.get(tid, "")}
        for tid in terminal_stock
    }
    return part_terminals, terminal_stock, unavailable_set, terminal_detail

part_terminals, terminal_stock, unavailable_terminals, terminal_detail = \
    _build_terminal_structures(_terminal_parts_raw, _terminal_stock_raw)

terminal_running_stock = dict(terminal_stock)

def _reset_terminal_running_stock():
    global terminal_running_stock
    terminal_running_stock = dict(terminal_stock)

def _consume_terminals(part, qty_produced):
    """Consume terminal stock when producing qty_produced pieces of part."""
    for tid in part_terminals.get(part, []):
        if tid in terminal_running_stock:
            terminal_running_stock[tid] = max(0.0, terminal_running_stock[tid] - qty_produced)


def terminal_available_qty_for_part(part, use_running_stock=False):
    """
    Returns the maximum number of pieces of `part` that can be produced
    given the available terminal stock (1 terminal consumed per piece).
    Returns float('inf') if part has no terminal requirements.
    """
    required = part_terminals.get(part, [])
    if not required:
        return float("inf")
    stock = terminal_running_stock if use_running_stock else terminal_stock
    min_stock = min(stock.get(tid, 0.0) for tid in required)
    return max(0.0, min_stock)


def terminal_availability_check(part, category, use_running_stock=False):
    """
    Checks whether a part can be scheduled today given terminal constraints.

    RUNNER:
      - All required terminals must have stock >= 85% of daily_indent
      - AND stock must cover at least MIN_RUN_HOURS of production
      - Returns (False, reason) if either check fails for any terminal

    REPEATER / STRANGER:
      - Terminals must cover the FULL daily indent (not just min run hours)
      - If max producible < daily_indent → BLOCKED
      - Returns (False, reason) with explanation

    Returns: (allowed: bool, reason: str, allowed_qty: float)
      allowed_qty = how many pieces can actually be produced (capped by terminals)
    """
    if not _terminal_data_loaded:
        return True, "Terminal data not loaded", float("inf")

    required = part_terminals.get(part, [])
    if not required:
        return True, "No terminal constraint", float("inf")

    stock  = terminal_running_stock if use_running_stock else terminal_stock
    daily  = indent_daily.get(part, 0.0)
    r_val  = rate.get(part, 1.0)

    # Find limiting terminal (minimum stock)
    min_stock     = float("inf")
    limiting_term = "—"
    for tid in required:
        s = stock.get(tid, 0.0)
        if s < min_stock:
            min_stock     = s
            limiting_term = tid

    max_producible = min_stock  # 1 terminal per piece → max qty = min stock

    # ── RUNNER logic ────────────────────────────────────────────
    if category == "Runner":
        threshold_85 = (RUNNER_TERMINAL_MIN_PCT / 100.0) * daily
        for tid in required:
            s = stock.get(tid, 0.0)
            # Check 1: each terminal must have >= 85% of daily indent
            if s < threshold_85:
                return (False,
                        f"Runner blocked — terminal {tid} stock={s:.0f} < "
                        f"85% of daily_indent ({threshold_85:.0f} pcs)",
                        0.0)
            # Check 2: must be enough for at least MIN_RUN_HOURS production
            min_run_qty = MIN_RUN_HOURS * r_val
            if s < min_run_qty:
                return (False,
                        f"Runner blocked — terminal {tid} stock={s:.0f} < "
                        f"min run qty ({min_run_qty:.0f} pcs = {MIN_RUN_HOURS}h × {r_val:.0f}/h)",
                        0.0)
        # All terminals pass → allowed qty is min stock (but runner usually has plenty)
        return True, f"Runner terminals OK (limiting: {limiting_term} stock={min_stock:.0f})", max_producible

    # ── REPEATER / STRANGER logic ────────────────────────────────
    if max_producible <= 0:
        return (False,
                f"BLOCKED — terminal {limiting_term} has zero stock",
                0.0)

    if daily > 0 and max_producible < daily:
        # Check if even min run hours is possible
        min_run_qty = MIN_RUN_HOURS * r_val
        if max_producible < min_run_qty:
            return (False,
                    f"BLOCKED — terminal {limiting_term} stock={max_producible:.0f} covers only "
                    f"{max_producible/r_val:.2f}h < {MIN_RUN_HOURS}h minimum run "
                    f"(daily_indent={daily:.0f})",
                    0.0)
        # Terminals cover some but NOT full daily indent → BLOCKED for Repeater/Stranger
        # We require full daily indent to be met for these categories
        return (False,
                f"BLOCKED — terminal {limiting_term} stock={max_producible:.0f} covers only "
                f"{max_producible:.0f} pcs but daily indent = {daily:.0f} pcs "
                f"(terminals must cover full daily indent for {category})",
                0.0)

    # max_producible >= daily → fully allowed
    return (True,
            f"{category} terminals OK (limiting: {limiting_term} "
            f"stock={min_stock:.0f} >= daily_indent={daily:.0f})",
            max_producible)


def terminal_criticality(part, use_running_stock=False):
    required = part_terminals.get(part, [])
    if not required:
        return "OK", "—", float("inf")
    daily = indent_daily.get(part, 0.0)
    stock = terminal_running_stock if use_running_stock else terminal_stock
    min_stock     = float("inf")
    limiting_term = "—"
    for tid in required:
        s = stock.get(tid, float("inf"))
        if s < min_stock:
            min_stock     = s
            limiting_term = tid
    if min_stock == float("inf"):
        return "OK", "—", float("inf")
    if min_stock <= 0:
        return "BLOCKED", limiting_term, 0.0
    if daily > 0:
        cov_pct = (min_stock / daily) * 100
        if cov_pct < TERMINAL_HIGHLY_CRITICAL_PCT:
            return "HIGHLY CRITICAL", limiting_term, min_stock
        if cov_pct < TERMINAL_CRITICAL_PCT:
            return "CRITICAL", limiting_term, min_stock
    return "OK", limiting_term, min_stock


def is_terminal_blocked(part):
    """Quick gate check: returns (blocked, reason)."""
    if not _terminal_data_loaded:
        return False, ""
    allowed, reason, _ = terminal_availability_check(
        part, part_category.get(part, "Stranger"), use_running_stock=False)
    if not allowed:
        return True, reason
    return False, ""


def terminals_for_display(part):
    req = part_terminals.get(part, [])
    if not req:
        return "—"
    return ", ".join(
        f"{t}(stk={terminal_stock.get(t,'?'):.0f})" if t in terminal_stock else t
        for t in req
    )

def blocked_terminals_for_display(part):
    req     = part_terminals.get(part, [])
    blocked = [t for t in req if terminal_stock.get(t, float("inf")) <= 0]
    return ", ".join(blocked) or "—"

_parts_with_terminals = sum(1 for t in part_terminals.values() if t)
_parts_no_terminals   = sum(1 for t in part_terminals.values() if not t)
_n_blocked_terminals  = len(unavailable_terminals)

print(f"\n  Terminal system  (1 terminal consumed per piece):")
if part_terminals:
    print(f"    Parts mapped to terminals : {_parts_with_terminals}")
    print(f"    Parts with no terminals   : {_parts_no_terminals}  (unconstrained)")
    print(f"    Terminals with zero stock : {_n_blocked_terminals}")
    if unavailable_terminals:
        for tid in sorted(unavailable_terminals):
            affected = [p for p, ts in part_terminals.items() if tid in ts]
            print(f"      ✗  {tid:<22} stock=0  →  blocks {len(affected)} part(s)")
else:
    print(f"    No terminal data — constraint inactive")

# =============================================================
# OPD CAP
# =============================================================

def part_opd_cap(part, current_inventory_dict=None):
    inv   = (current_inventory_dict or inventory).get(part, 0.0)
    daily = indent_daily.get(part, 0.0)
    days  = inv / daily if daily > 0 else 0.0
    if days < 1.0:
        return OPD_SCENARIO_0
    else:
        return OPD_SCENARIO_1


def stranger_cap_qty(part, current_inventory_dict):
    daily = indent_daily.get(part, 0.0)
    r_val = rate.get(part, 1.0)
    inv   = current_inventory_dict.get(part, 0.0)
    if daily <= 0:
        return 0.0
    multiplier  = STRANGER_BASE_MULTIPLIER * (STRANGER_RATE_ANCHOR / r_val) if r_val > 0 else STRANGER_BASE_MULTIPLIER
    raw_cap     = daily * multiplier
    opd_ceiling = part_opd_cap(part, current_inventory_dict) * daily
    cap         = max(daily, min(raw_cap, opd_ceiling))
    return max(0.0, cap - inv)


def runner_should_soft_defer(part, machine_hours_dict):
    inv   = inventory.get(part, 0.0)
    daily = indent_daily.get(part, 0.0)
    if daily <= 0:
        return False
    inv_days = inv / daily
    if inv_days < 1.0 or inv_days >= SAFETY_DAYS:
        return False
    compatible = vt_compat.get(part, [])
    if not compatible:
        return True
    threshold_hrs = AVAILABLE_HOURS * (RUNNER_DEFER_MACHINE_UTIL_PCT / 100.0)
    for m in compatible:
        if machine_hours_dict.get(m, 0.0) < threshold_hrs:
            return False
    return True


def should_skip(part):
    """
    Gate 4a : daily indent <= MIN_DAILY_INDENT                   → SKIP
    Gate 4b : whole monthly indent runs in <= MIN_INDENT_HOURS   → SKIP
    Gate 4c : terminal check fails (see terminal_availability_check) → BLOCKED
    Gate 5  : inventory >= SAFETY_DAYS × daily (3-day ceiling)  → AT CEILING
    """
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    # Gate 4c: terminal check using the full terminal_availability_check
    allowed, reason, _ = terminal_availability_check(
        part, part_category.get(part, "Stranger"), use_running_stock=False)
    if not allowed:
        return True, f"TERMINAL BLOCKED — {reason}"

    inv = inventory.get(part, 0.0)
    if daily > 0 and inv >= SAFETY_DAYS * daily:
        return True, (f"Inventory ({inv:.0f}) ≥ {SAFETY_DAYS}-day ceiling "
                      f"({SAFETY_DAYS * daily:.0f} pcs) — at ceiling, skip today")

    return False, ""


def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print(f"  WARNING: VT_Machine_Part_Count sheet missing 'Machine' or 'Part_Count' column")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category = build_category(vt_parts_raw)

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)
    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"
    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{SAFETY_DAYS} days safety floor)"

def opd_cap(scenario_id):
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_1)

def _inv_scenario_label(part, inv_dict):
    inv   = inv_dict.get(part, 0.0)
    daily = indent_daily.get(part, 0.0)
    days  = inv / daily if daily > 0 else 0.0
    if days < 1:             return "S0-CRITICAL"
    elif days < SAFETY_DAYS: return "S1-BELOW_SAFETY"
    else:                    return "S2-AT_CEILING"

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, SAFETY_DAYS - days_cov) / SAFETY_DAYS)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})
    if not rows:
        return {}, []
    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        # Get terminal availability info for display
        t_blocked, t_reason = is_terminal_blocked(p)
        _, _, term_max_prod = terminal_criticality(p)
        term_level, limiting_term, _ = terminal_criticality(p)
        inv_val = r["inv"]
        daily_v = r["daily"]
        days_v  = inv_val / daily_v if daily_v > 0 else 0
        score_rows.append({
            "Part":               p,
            "Category":           r["cat"],
            "Color":              part_color.get(p, "UNKNOWN"),
            "Tools":              tools_available.get(p, 1),
            "Required_Terminals": terminals_for_display(p),
            "Blocked_Terminals":  blocked_terminals_for_display(p),
            "Terminal_Status":    "BLOCKED" if t_blocked else "OK",
            "Terminal_Block_Reason": t_reason if t_blocked else "",
            "Fixed_Machine":      ", ".join(get_fixed_machines_for_part(p)) or "—",
            "Inventory_Now":      round(r["inv"], 0),
            "Daily_Indent":       round(r["daily"], 2),
            "Days_Coverage":      round(r["days_cov"], 2),
            "Inv_Scenario":       _inv_scenario_label(p, inventory),
            "OPD_Cap":            part_opd_cap(p),
            "Safety_Ceiling":     SAFETY_DAYS,
            "Buffer_Status":      (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "AT_CEILING"
            ),
            "Urgency_Score":   round(urgency_score, 1),
            "Category_Score":  category_score,
            "Indent_Score":    round(indent_score, 1),
            "Final_Score":     round(final_score, 2),
        })
    return scores, score_rows

# =============================================================
# DISPLACEMENT PASS
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part, current_inventory,
                           plan, already_planned, priority_scores):
    daily    = indent_daily.get(part, 0)
    r_val    = rate.get(part, 1)
    category = part_category.get(part, "Stranger")
    score    = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])
    if not compatible:
        return False
    candidate_machines = [
        m for m in compatible
        if round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]
    if not candidate_machines:
        return False
    best_machine     = None
    best_victim_row  = None
    best_victim_days = -1
    for m in candidate_machines:
        # Cannot displace on a fixed machine that is reserved for another fixed part
        if is_fixed_machine(m) and part not in vt_fixed_map.get(m, []):
            if not fixed_machine_is_open(m, current_inventory):
                continue
        for row in [r for r in plan if r["Machine"] == m]:
            vpart  = row["Part"]
            if part_category.get(vpart, "Stranger") == "Runner":
                continue
            vdaily = indent_daily.get(vpart, 0)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m
    if best_machine is None or best_victim_row is None:
        return False
    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)
    best_victim_row["Run_Hours"]      = round(float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) + float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"] = str(best_victim_row.get("Type", "Primary")) + " [DISPLACED]"
    current_inventory[vpart]    = round(current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - reduce_h, 4)
    last = machine_last_part.get(best_machine)
    if last is None or last == part:
        co_hrs = 0.0
    else:
        base_co    = vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        last_color = part_color.get(last, "UNKNOWN")
        new_color  = part_color.get(part, "UNKNOWN")
        purge      = (COLOR_PURGE_HRS if last_color != new_color
                      and last_color != "UNKNOWN" and new_color != "UNKNOWN" else 0.0)
        co_hrs = base_co + purge
    eff_free = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    run_hrs  = max(MIN_RUN_HOURS, min(eff_free, MIN_RUN_HOURS))
    qty      = round(run_hrs * r_val, 0)
    machine_hours[best_machine]     = round(machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)
    _consume_terminals(part, qty)
    plan.append({
        "Part":             part,
        "Color":            part_color.get(part, "UNKNOWN"),
        "Category":         category,
        "Machine":          best_machine,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) else "No",
        "Type":             "Displacement [ZERO-INV PRIORITY]",
        "Role":             "Primary",
        "Tools_Available":  tools_available.get(part, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if qty >= daily else "NO — partial",
        "Stagger_Adjusted": "No",
        "Displaced_Victim": vpart,
        "Victim_Days_Stock":round(best_victim_days, 2),
        "Fixed_Machine_Used": "YES" if is_fixed_machine(best_machine) and part in vt_fixed_map.get(best_machine, []) else "No",
    })
    print(f"      ↳ DISPLACEMENT  {part:26s} → {best_machine:15s}  "
          f"freed from {vpart} ({best_victim_days:.1f}d)  run={run_hrs:.2f}h  qty={qty:.0f}")
    return True

# =============================================================
# MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours, machine_last_part, inv_days):
    """
    Ranks machines for a part.
    Fixed machines for this part are sorted FIRST (highest preference).
    Non-fixed machines come after.
    Within each group, sorted by cost (changeover, utilisation, etc.)
    """
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)
    new_color   = part_color.get(part, "UNKNOWN")
    fixed_for_part = set(get_fixed_machines_for_part(part))

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue

        # ── Fixed machine guard ──────────────────────────────────
        # If this machine is fixed for OTHER parts (not this part):
        #   → only allowed if all fixed parts on it have >= 3-day inventory (machine is open)
        if is_fixed_machine(m) and m not in fixed_for_part:
            if not fixed_machine_is_open(m, machine_hours):
                # Use current_inventory proxy via machine_hours context
                # We'll do a proper check in assign_part; here just skip if reserved
                continue

        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = part_color.get(last, "UNKNOWN")
            same_color = (last_color == new_color and last_color != "UNKNOWN" and new_color != "UNKNOWN")
            purge      = 0.0 if same_color else (
                COLOR_PURGE_HRS if last_color != "UNKNOWN" and new_color != "UNKNOWN" else 0.0)
            co_hrs      = base_co + purge
            color_bonus = -0.08 if same_color else 0.0

        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        # Strong bonus for fixed machine (negative cost = preferred)
        fixed_bonus     = -0.50 if m in fixed_for_part else 0.0
        cost = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus + fixed_bonus
        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock


def _co_hrs_for(p, m, machine_last_part):
    last = machine_last_part.get(m)
    if last is None or last == p:
        return 0.0
    base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
    last_color = part_color.get(last, "UNKNOWN")
    new_color  = part_color.get(p,    "UNKNOWN")
    purge      = (COLOR_PURGE_HRS if last_color != new_color
                  and last_color != "UNKNOWN" and new_color != "UNKNOWN" else 0.0)
    return base_co + purge

# =============================================================
# ASSIGN PART
# Terminal-aware qty capping:
#   - qty produced on any machine/tool is capped to terminal_available_qty_for_part()
#   - _consume_terminals() is called after each assignment
# Fixed machine preference:
#   - rank_machines() gives strong preference to fixed machines for fixed parts
#   - Non-fixed machines of a reserved fixed machine are excluded
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores):
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    inv_days = inv_now / daily if daily > 0 else 999

    compatible = vt_compat.get(part, [])
    if not compatible:
        return []

    # Fixed machine: put fixed machines for this part first in the candidate list
    fixed_for_part = get_fixed_machines_for_part(part)
    if fixed_for_part:
        other_machines = [m for m in compatible if m not in fixed_for_part]
        compatible = fixed_for_part + other_machines

    # Get terminal availability (use running stock for live scheduling)
    term_level, limiting_term, term_max_prod = terminal_criticality(part, use_running_stock=True)
    t_allowed, t_reason, t_allowed_qty = terminal_availability_check(
        part, category, use_running_stock=True)

    if not t_allowed:
        # Should not happen here (should_skip catches this) but belt-and-suspenders
        return []

    # Cap producible qty by terminals
    if t_allowed_qty == float("inf"):
        terminal_cap_qty = float("inf")
    else:
        terminal_cap_qty = t_allowed_qty

    # Single tool rule: Repeater and Stranger → single machine always
    is_single_machine_only = (category in ("Repeater", "Stranger"))

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    # Cap shortfall by terminal availability
    if terminal_cap_qty != float("inf"):
        capped_shortfall = min(total_shortfall, terminal_cap_qty)
        if capped_shortfall < total_shortfall:
            hrs_for_full = max(MIN_RUN_HOURS, capped_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
        total_shortfall = capped_shortfall

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    ranked, runner_lock = rank_machines(part, compatible, machine_hours, machine_last_part, inv_days)
    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    # Cap run hours by remaining terminal stock on primary machine
    remaining_terminal = terminal_available_qty_for_part(part, use_running_stock=True)
    max_qty_from_terminal = min(remaining_terminal, total_shortfall) if remaining_terminal != float("inf") else total_shortfall
    run1 = min(eff1, hrs_for_full, max_qty_from_terminal / r_val if r_val > 0 else eff1)
    run1 = max(run1, MIN_RUN_HOURS)
    # Final cap: don't exceed what terminals allow
    if remaining_terminal != float("inf"):
        run1 = min(run1, remaining_terminal / r_val if r_val > 0 else run1)
    qty1 = min(round(run1 * r_val, 0), remaining_terminal if remaining_terminal != float("inf") else float("inf"))
    qty1 = round(qty1, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)
    _consume_terminals(part, qty1)

    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    last_m1       = machine_state.get(m1)
    purge_applied = (last_m1 is not None and last_m1 != part
                     and part_color.get(last_m1, "UNKNOWN") != color
                     and part_color.get(last_m1, "UNKNOWN") != "UNKNOWN"
                     and color != "UNKNOWN")

    is_fixed_machine_used = m1 in fixed_for_part

    new_rows.append({
        "Part":             part,
        "Color":            color,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Color_Purge":      "Yes" if purge_applied else "No",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Stagger_Adjusted": "No",
        "Inv_Scenario":     _inv_scenario_label(part, inventory),
        "Required_Terminals":   terminals_for_display(part),
        "Terminal_Criticality": term_level,
        "Limiting_Terminal":    limiting_term,
        "Terminal_Max_Prod":    round(terminal_cap_qty, 0) if terminal_cap_qty != float("inf") else "Unlimited",
        "Fixed_Machine_Used":   "YES" if is_fixed_machine_used else "No",
        "Fixed_Machine_Assigned": ", ".join(fixed_for_part) if fixed_for_part else "—",
    })

    # PHASE 2: Tool expansion — RUNNERS ONLY
    if not indent_met_on_primary and not is_single_machine_only:
        is_critical = (inv_now == 0)
        if is_critical:
            tool_hard_cap = tools
        else:
            free_on_m1 = round(AVAILABLE_HOURS - machine_hours.get(m1, 0), 4)
            total_can_produce_1tool = qty1 + (free_on_m1 * r_val)
            if total_can_produce_1tool >= total_shortfall - 0.5:
                tool_hard_cap = 1
            else:
                remaining_after_m1 = total_shortfall - produced_so_far
                max_per_machine = AVAILABLE_HOURS * r_val
                extra_machines_needed = math.ceil(remaining_after_m1 / max_per_machine)
                tool_hard_cap = min(tools, 1 + extra_machines_needed)

        while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(MIN_RUN_HOURS, shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)
            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(part, remaining_machines, machine_hours,
                                           machine_last_part, inv_days)
            if not ranked_next:
                break
            mx, cox, effx, _ = ranked_next[0]
            # Cap by remaining terminal stock
            rem_term = terminal_available_qty_for_part(part, use_running_stock=True)
            run_x    = min(effx, hrs_needed)
            run_x    = max(run_x, MIN_RUN_HOURS)
            if rem_term != float("inf"):
                run_x = min(run_x, rem_term / r_val if r_val > 0 else run_x)
            qty_x = min(round(run_x * r_val, 0), rem_term if rem_term != float("inf") else float("inf"))
            qty_x = round(qty_x, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)
            _consume_terminals(part, qty_x)
            indent_met_here = (produced_so_far >= total_shortfall - 0.5)
            last_mx    = machine_state.get(mx)
            purge_x    = (last_mx is not None and last_mx != part
                         and part_color.get(last_mx, "UNKNOWN") != color
                         and part_color.get(last_mx, "UNKNOWN") != "UNKNOWN"
                         and color != "UNKNOWN")
            new_rows.append({
                "Part":             part,
                "Color":            color,
                "Category":         category,
                "Machine":          mx,
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Color_Purge":      "Yes" if purge_x else "No",
                "Type":             "Tool-Expansion (min tools)",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tool_hard_cap})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
                "Inv_Scenario":     _inv_scenario_label(part, inventory),
                "Required_Terminals":   terminals_for_display(part),
                "Terminal_Criticality": term_level,
                "Limiting_Terminal":    limiting_term,
                "Terminal_Max_Prod":    round(terminal_cap_qty, 0) if terminal_cap_qty != float("inf") else "Unlimited",
                "Fixed_Machine_Used":   "YES" if mx in fixed_for_part else "No",
                "Fixed_Machine_Assigned": ", ".join(fixed_for_part) if fixed_for_part else "—",
            })
            print(f"      ↳ TOOL-EXP(min) {part:22s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"{'COVERED ✓' if indent_met_here else 'still short'}")
    elif not indent_met_on_primary and is_single_machine_only:
        print(f"      ↳ SINGLE-TOOL [{category}] {part:22s} → partial accepted "
              f"qty={qty1:.0f}  shortfall={(total_shortfall - qty1):.0f}  (single tool rule)")

    # PHASE 3: Inventory build — extend on primary machine only (gradual)
    if category == "Stranger":
        headroom_qty = stranger_cap_qty(part, current_inventory)
    else:
        cap_days     = part_opd_cap(part, current_inventory)
        inv_after    = current_inventory.get(part, 0)
        headroom_qty = max(0.0, cap_days * daily - inv_after)

    # Cap headroom by remaining terminal stock
    rem_term_phase3 = terminal_available_qty_for_part(part, use_running_stock=True)
    if rem_term_phase3 != float("inf"):
        headroom_qty = min(headroom_qty, rem_term_phase3)

    if headroom_qty > 0:
        row = new_rows[0]  # primary machine only
        m   = row["Machine"]
        free_m = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if free_m >= 0.05:
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs >= 0.05:
                extra_qty = round(extend_hrs * r_val, 0)
                # Final terminal cap
                if rem_term_phase3 != float("inf"):
                    extra_qty = min(extra_qty, rem_term_phase3)
                    extend_hrs = extra_qty / r_val if r_val > 0 else extend_hrs
                row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
                row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
                row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
                row["Type"]           = str(row["Type"]) + "+InvBuild(gradual)"
                machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
                current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
                _consume_terminals(part, extra_qty)
                inv_now_after = current_inventory.get(part, 0)
                days_now      = inv_now_after / daily if daily > 0 else 0
                print(f"      ↳ INV-BUILD(grad) {part:22s} on {m:15s}  "
                      f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}  "
                      f"[3×cap, {days_now:.1f}d → gradual]")

    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# TOOL-CHANGER HELPERS
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CO: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra

MIN_CO_GAP_HRS = 20 / 60.0

def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def _last_row_before_co(ev, plan):
    return ev["row_before"]

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0
    adjustments = 0
    max_passes  = len(events) * 2
    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])
        conflict = None
        for i in range(len(events) - 1):
            co_dur_e = events[i]["co_duration"]
            required = co_dur_e + MIN_CO_GAP_HRS
            gap      = events[i+1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break
        if conflict is None:
            break
        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap
        row_late  = _last_row_before_co(ev_late, plan)
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = indent_daily.get(p_late, 0)
        inv_l     = current_inventory.get(p_late, 0)
        cap_qty   = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom  = max(0.0, cap_qty - inv_l)
        push_hrs  = shortfall_hrs
        push_qty  = round(push_hrs * r_late, 0)
        used_m    = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
                        for r in plan if r["Machine"] == m_late)
        free_m    = max(0.0, AVAILABLE_HOURS - used_m)
        can_push  = (push_qty <= headroom and push_hrs <= free_m + 0.001 and r_late > 0)
        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_late] = round(current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue
        row_early  = _last_row_before_co(ev_early, plan)
        p_early    = row_early["Part"]
        r_early    = rate.get(p_early, 1)
        daily_e    = indent_daily.get(p_early, 0)
        inv_e      = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)
        can_pull   = (pull_qty <= max_pull and r_early > 0
                      and produced_e - pull_qty >= MIN_RUN_HOURS * r_early)
        if can_pull:
            row_early["Run_Hours"]      = round(float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_early] = round(current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break
    return adjustments

def stagger_changeovers_serial_queue(plan, machines):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return
    events.sort(key=lambda e: e["natural_start"])
    print(f"  {len(events)} CO events queued")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6} {'Purge':>6}")
    print(f"  {'─'*105}")
    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0
    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h
        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60
        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs
        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        fill_str  = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"
        before_color = part_color.get(ev["part_before"], "?")
        after_color  = part_color.get(ev["part_after"],  "?")
        purge_str    = "PURGE" if before_color != after_color and before_color not in ("?","UNKNOWN") and after_color not in ("?","UNKNOWN") else "—"
        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6} "
              f"{purge_str:>6}")
    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. TC free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events waited: {n_waited}/{len(events)}  |  Extra pcs: {total_extra_pcs:,.0f}")

def stagger_changeovers(plan, machines):
    stagger_changeovers_serial_queue(plan, machines)

# =============================================================
# 22H UTILIZATION ENFORCER — Runner >>>> Repeater >>>> Stranger
# STRICT ORDER: Runner > Repeater > Stranger on every machine
# Fixed machine guard enforced in all steps
# =============================================================

def _is_machine_available_for_part(m, part, current_inventory):
    """
    Returns True if this machine can be used for this part.
    Enforces fixed machine rules:
      - If machine is fixed for OTHER parts AND those parts are below 3-day floor → False
      - If machine is fixed for THIS part → True (preferred)
      - If machine is not fixed → True
    """
    fixed_for_part = get_fixed_machines_for_part(part)
    if is_fixed_machine(m):
        if m in fixed_for_part:
            return True  # this is the designated machine for this part — always OK
        # machine is fixed for other parts → only allow if all those fixed parts are at ceiling
        return fixed_machine_is_open(m, current_inventory)
    return True  # non-fixed machine → always available


def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val   = rate.get(p, 1)
    # Cap qty by terminal stock
    rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
    qty      = round(run_hrs * r_val, 0)
    if rem_term != float("inf"):
        qty = min(qty, rem_term)
    qty = round(qty, 0)
    last    = machine_last_part.get(m)
    p_color = part_color.get(p,    "UNKNOWN")
    l_color = part_color.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != p
                 and p_color != l_color
                 and p_color != "UNKNOWN" and l_color != "UNKNOWN")
    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)
    _consume_terminals(p, qty)
    fixed_for_p = get_fixed_machines_for_part(p)
    plan.append({
        "Part":             p,
        "Color":            p_color,
        "Category":         part_category.get(p, "Stranger"),
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if has_purge else "No",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
        "Inv_Scenario":       _inv_scenario_label(p, inventory),
        "Required_Terminals": terminals_for_display(p),
        "Fixed_Machine_Used": "YES" if m in fixed_for_p else "No",
        "Fixed_Machine_Assigned": ", ".join(fixed_for_p) if fixed_for_p else "—",
    })
    return qty


def _get_candidates_by_tier(tier_category, machine, machine_last_part,
                             all_parts, already_planned, current_inventory,
                             plan, _m_is_fixed_and_reserved):
    """
    Returns unplanned parts of a specific category that are compatible with this machine.
    Fixed machine guard enforced: if machine is reserved for fixed parts,
    only those fixed parts (which must be Runners) are allowed.
    """
    last_on_m  = machine_last_part.get(machine)
    last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

    candidates = []
    for p in all_parts:
        if part_category.get(p, "Stranger") != tier_category:
            continue
        if p in already_planned:
            continue
        if machine not in vt_compat.get(p, []):
            continue
        if should_skip(p)[0]:
            continue
        if indent_monthly.get(p, 0) <= 0:
            continue
        if rate.get(p, 0) <= 0:
            continue
        inv_now  = current_inventory.get(p, 0)
        daily_p  = indent_daily.get(p, 0)
        cap_days = part_opd_cap(p, current_inventory)
        if inv_now >= cap_days * daily_p:
            continue
        # Fixed machine guard
        if not _is_machine_available_for_part(machine, p, current_inventory):
            continue
        candidates.append(p)

    def _sort_key(p):
        needs_co   = 0 if (last_on_m is None or last_on_m == p) else 1
        p_col      = part_color.get(p, "UNKNOWN")
        same_color = (0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1)
        inv_now    = current_inventory.get(p, 0)
        is_crit    = 1 if inv_now == 0 else 0
        sc         = priority_scores_global.get(p, 0)
        return (needs_co, same_color, -is_crit, -sc)

    candidates.sort(key=_sort_key)
    return candidates


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):
    """
    Minimum utilisation floor: UTIL_TARGET_PCT.
    STRICT ORDER per machine: Runner >>>> Repeater >>>> Stranger.
    Fixed machine guard enforced at every step.
    Terminal availability checked before adding any part.
    """
    print(f"\n  22H UTILIZATION ENFORCER  (Runner >>>> Repeater >>>> Stranger on every machine)")
    micro_idle_log = []
    floor_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0] and rate.get(p, 0) > 0 and indent_monthly.get(p, 0) > 0
    ]

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # Is this machine reserved for its fixed parts?
        _m_is_fixed_and_reserved = (
            is_fixed_machine(m) and not fixed_machine_is_open(m, current_inventory)
        )

        # ── STEP 0: 90% floor — extend existing parts ───────────────
        current_util_hrs = machine_hours.get(m, 0)
        if current_util_hrs < floor_hrs:
            needed = round(floor_hrs - current_util_hrs, 4)
            parts_on_m = [row for row in plan if row["Machine"] == m]
            if parts_on_m:
                # Extend by priority: Runner first, then Repeater, then Stranger
                parts_on_m_sorted = sorted(
                    parts_on_m,
                    key=lambda r: (
                        CATEGORY_TIER.get(part_category.get(r["Part"], "Stranger"), 2),
                        -priority_scores.get(r["Part"], 0)
                    )
                )
                for row in parts_on_m_sorted:
                    if needed <= 0.001:
                        break
                    p_ext     = row["Part"]
                    r_ext     = rate.get(p_ext, 1)
                    ext_hrs   = min(needed, remaining)
                    if ext_hrs < 0.001:
                        continue
                    # Check terminal headroom
                    rem_term = terminal_available_qty_for_part(p_ext, use_running_stock=True)
                    extra_qty = round(ext_hrs * r_ext, 0)
                    if rem_term != float("inf"):
                        extra_qty = min(extra_qty, rem_term)
                        if extra_qty <= 0:
                            continue
                        ext_hrs = extra_qty / r_ext if r_ext > 0 else ext_hrs
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Floor90"
                    machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                    current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                    _consume_terminals(p_ext, extra_qty)
                    remaining               = round(remaining - ext_hrs, 4)
                    needed                  = round(needed - ext_hrs, 4)
                    new_util = round(machine_hours.get(m, 0) / AVAILABLE_HOURS * 100, 1)
                    print(f"    [S0-FLOOR90] {p_ext:28s} on {m:15s}  "
                          f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  util now {new_util}%")

        if remaining < 0.05:
            continue

        # ── STEP 1: Extend existing parts (OPD-capped 3×) ────────────
        # Runner first, then Repeater, then Stranger
        parts_on_machine = sorted(
            list({row["Part"] for row in plan if row["Machine"] == m}),
            key=lambda p: (
                CATEGORY_TIER.get(part_category.get(p, "Stranger"), 2),
                -priority_scores.get(p, 0)
            )
        )
        for p in parts_on_machine:
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_days = part_opd_cap(p, current_inventory)
            headroom = max(0.0, cap_days * daily_p - inv_now)
            # Cap by terminal stock
            rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
            if rem_term != float("inf"):
                headroom = min(headroom, rem_term)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            if rem_term != float("inf"):
                extra_qty = min(extra_qty, rem_term)
                if extra_qty <= 0:
                    continue
                ext_hrs = extra_qty / r_val if r_val > 0 else ext_hrs
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            _consume_terminals(p, extra_qty)
            remaining            = round(remaining - ext_hrs, 4)
            cat_label = part_category.get(p, "Stranger")
            print(f"    [S1-EXTEND]  {p:28s} [{cat_label:8s}] on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 2: Unplanned RUNNERS — always first ─────────────────
        runner_candidates = _get_candidates_by_tier(
            "Runner", m, machine_last_part, all_parts, already_planned,
            current_inventory, plan, _m_is_fixed_and_reserved)

        for p in runner_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            # Terminal check
            t_ok, t_rsn, t_qty = terminal_availability_check(p, "Runner", use_running_stock=True)
            if not t_ok:
                print(f"    [S2-RUNNER]  {p:28s} → SKIP (terminal: {t_rsn})")
                continue
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_days = part_opd_cap(p, current_inventory)
            headroom = max(0.0, cap_days * daily_p - inv_now)
            rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
            if rem_term != float("inf"):
                headroom = min(headroom, rem_term)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = min(eff_free, max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            if rem_term != float("inf"):
                run_hrs = min(run_hrs, rem_term / r_val if r_val > 0 else run_hrs)
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Runner", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-RUNNER]  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  [RUNNER PRIORITY]")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 3: Unplanned REPEATERS — second ─────────────────────
        repeater_candidates = _get_candidates_by_tier(
            "Repeater", m, machine_last_part, all_parts, already_planned,
            current_inventory, plan, _m_is_fixed_and_reserved)

        for p in repeater_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            # Terminal check — Repeater must have full daily indent coverage
            t_ok, t_rsn, t_qty = terminal_availability_check(p, "Repeater", use_running_stock=True)
            if not t_ok:
                print(f"    [S3-REPEAT]  {p:28s} → SKIP (terminal: {t_rsn})")
                continue
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_days = part_opd_cap(p, current_inventory)
            headroom = max(0.0, cap_days * daily_p - inv_now)
            rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
            if rem_term != float("inf"):
                headroom = min(headroom, rem_term)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = min(eff_free, max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            if rem_term != float("inf"):
                run_hrs = min(run_hrs, rem_term / r_val if r_val > 0 else run_hrs)
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Repeater", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S3-REPEAT]  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 4: Unplanned STRANGERS — last ───────────────────────
        stranger_candidates = _get_candidates_by_tier(
            "Stranger", m, machine_last_part, all_parts, already_planned,
            current_inventory, plan, _m_is_fixed_and_reserved)

        for p in stranger_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            # Terminal check — Stranger must have full daily indent coverage
            t_ok, t_rsn, t_qty = terminal_availability_check(p, "Stranger", use_running_stock=True)
            if not t_ok:
                print(f"    [S4-STRANGER] {p:27s} → SKIP (terminal: {t_rsn})")
                continue
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = stranger_cap_qty(p, current_inventory)
            rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
            if rem_term != float("inf"):
                headroom = min(headroom, rem_term)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = min(eff_free, max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            if rem_term != float("inf"):
                run_hrs = min(run_hrs, rem_term / r_val if r_val > 0 else run_hrs)
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Stranger", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S4-STRANGER] {p:27s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 5: Re-run planned Runners (spare tool) ──────────────
        planned_runners_elsewhere = [
            p for p in already_planned
            if part_category.get(p, "Stranger") == "Runner"
            and m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
            and tools_available.get(p, 1) > len({row["Machine"] for row in plan if row["Part"] == p})
            and _is_machine_available_for_part(m, p, current_inventory)
        ]
        planned_runners_elsewhere.sort(key=lambda p: -priority_scores.get(p, 0))

        for p in planned_runners_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            t_ok, t_rsn, t_qty = terminal_availability_check(p, "Runner", use_running_stock=True)
            if not t_ok:
                continue
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_days = part_opd_cap(p, current_inventory)
            headroom = max(0.0, cap_days * daily_p - inv_now)
            rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
            if rem_term != float("inf"):
                headroom = min(headroom, rem_term)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            run_hrs   = min(eff_free, max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            if rem_term != float("inf"):
                run_hrs = min(run_hrs, rem_term / r_val if r_val > 0 else run_hrs)
            qty = round(run_hrs * r_val, 0)
            if rem_term != float("inf"):
                qty = min(qty, rem_term)
            qty = round(qty, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            _consume_terminals(p, qty)
            remaining            = round(remaining - co_hrs - run_hrs, 4)
            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            p_color    = part_color.get(p, "UNKNOWN")
            last_p     = machine_state.get(m)
            lc         = part_color.get(last_p, "UNKNOWN") if last_p else "UNKNOWN"
            has_purge  = (last_p is not None and last_p != p
                          and p_color != lc and p_color != "UNKNOWN" and lc != "UNKNOWN")
            fixed_for_p = get_fixed_machines_for_part(p)
            plan.append({
                "Part":             p,
                "Color":            p_color,
                "Category":         "Runner",
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Color_Purge":      "Yes" if has_purge else "No",
                "Type":             "Re-run Runner (spare tool)",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
                "Inv_Scenario":       _inv_scenario_label(p, inventory),
                "Required_Terminals": terminals_for_display(p),
                "Fixed_Machine_Used": "YES" if m in fixed_for_p else "No",
                "Fixed_Machine_Assigned": ", ".join(fixed_for_p) if fixed_for_p else "—",
            })
            print(f"    [S5-RUNNER-RERUN] {p:24s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  tool {tools_used_now}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 6: Skipped parts last resort — Runner > Repeater > Stranger
        for tier_cat in ["Runner", "Repeater", "Stranger"]:
            if remaining < MIN_RUN_HOURS:
                break
            skipped_candidates = [
                p for p in all_skipped
                if part_category.get(p, "Stranger") == tier_cat
                and m in vt_compat.get(p, []) and rate.get(p, 0) > 0
                and _is_machine_available_for_part(m, p, current_inventory)
            ]
            last_on_m  = machine_last_part.get(m)
            last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
            skipped_candidates.sort(key=lambda p: (
                0 if (last_on_m is None or last_on_m == p) else 1,
                0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
                1 if current_inventory.get(p, 0) == 0 else 0,
                -indent_daily.get(p, 0)
            ))
            for p in skipped_candidates:
                if remaining < MIN_RUN_HOURS:
                    break
                t_ok, t_rsn, t_qty = terminal_availability_check(p, tier_cat, use_running_stock=True)
                if not t_ok:
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                r_val   = rate.get(p, 1)
                run_hrs = min(eff_free, max(MIN_RUN_HOURS,
                              indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
                run_hrs = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
                rem_term = terminal_available_qty_for_part(p, use_running_stock=True)
                if rem_term != float("inf"):
                    run_hrs = min(run_hrs, rem_term / r_val if r_val > 0 else run_hrs)
                qty = _add_part_to_machine(
                    p, m, run_hrs, co_hrs,
                    machine_hours, machine_last_part, current_inventory,
                    already_planned, plan, priority_scores,
                    f"Filler-Skipped-{tier_cat}", "Primary")
                remaining = round(remaining - co_hrs - run_hrs, 4)
                print(f"    [S6-SKIP-{tier_cat[:3].upper()}] {p:26s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")
                break

        # ── STEP 7: Log micro-idle ────────────────────────────────────
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            "All options exhausted / all parts at 3-day ceiling or terminal blocked",
                })
                print(f"    [⚠ IDLE]     {m:15s}  {final_remaining:.2f}h idle ({util_final}%)")

    return micro_idle_log

# =============================================================
# OUTPUT VIEW BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()
    output_rows = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part":                   part,
                "Color":                  part_color.get(part, "UNKNOWN"),
                "Category":               part_category.get(part, "Stranger"),
                "Tools_Available":        tools_available.get(part, 1),
                "Machines_Used":          len(rows),
                "Machine":                row["Machine"],
                "Fixed_Machine_Used":     row.get("Fixed_Machine_Used", "No"),
                "Fixed_Machine_Assigned": row.get("Fixed_Machine_Assigned", "—"),
                "Role":                   row.get("Role", "Primary"),
                "Run_Hours":              round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs":         round(float(row.get("Changeover_Hrs", 0)), 2),
                "Color_Purge":            row.get("Color_Purge", "No"),
                "Production_Qty":         round(float(row["Production_Qty"]), 0),
                "Daily_Indent":           round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Terminal_Max_Prod":      row.get("Terminal_Max_Prod", "Unlimited"),
                "Type":                   row.get("Type", "—"),
            })
        output_rows.append({
            "Part":                   f"  ↳ TOTAL — {part}",
            "Color":                  part_color.get(part, "UNKNOWN"),
            "Category":               "—",
            "Tools_Available":        tools_available.get(part, 1),
            "Machines_Used":          len(rows),
            "Machine":                f"{len(rows)} machines",
            "Fixed_Machine_Used":     "—",
            "Fixed_Machine_Assigned": "—",
            "Role":                   "TOTAL",
            "Run_Hours":              round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":         round(sum(float(r.get("Changeover_Hrs", 0)) for r in rows), 2),
            "Color_Purge":            "—",
            "Production_Qty":         round(total_qty, 0),
            "Daily_Indent":           round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0),
            "Terminal_Max_Prod":      "—",
            "Type":                   "—",
        })
        output_rows.append({k: "" for k in output_rows[-1].keys()})
    return pd.DataFrame(output_rows)


def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0
        fixed_ms = get_fixed_machines_for_part(p)
        rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Tools_Available":    tools_available.get(p, 1),
            "Fixed_Machine":      ", ".join(fixed_ms) if fixed_ms else "—",
            "Machines":           ", ".join(dict.fromkeys(part_machines[p])),
            "Machines_Count":     len(set(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(monthly, 0),
            "Gap_vs_Daily":       gap,
            "Gap_Direction":      gap_dir,
            "Extra_Days_Stock":   extra_days,
            "Inventory_Before":   round(inv_b, 0),
            "Inventory_After":    inv_after,
            "Days_Coverage_After":round(inv_after / daily, 2) if daily > 0 else 0,
            "Required_Terminals": terminals_for_display(p),
        })
    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_before = round(inv_b     / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty  = round(SAFETY_DAYS * daily, 0)
        gap_qty     = round(target_qty - inv_after, 0)
        gap_days    = max(0, round(gap_qty / daily, 2) if daily > 0 else 0)
        cap         = opd_cap(scenario_id)
        max_prod    = round(cap * daily, 0)

        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        else:
            status = "AT_CEILING"

        net_gain = (cap - 1) * daily if daily > 0 else 0
        if gap_qty <= 0:
            est_days = "AT CEILING"
        elif net_gain <= 0:
            est_days = "N/A"
        else:
            est_days = str(math.ceil(gap_qty / net_gain)) + " days"

        skip, skip_reason = should_skip(p)
        t_blocked, t_reason = is_terminal_blocked(p)
        fixed_ms = get_fixed_machines_for_part(p)
        rows.append({
            "Part":                   p,
            "Color":                  part_color.get(p, "UNKNOWN"),
            "Category":               part_category.get(p, "Stranger"),
            "Tools":                  tools_available.get(p, 1),
            "Fixed_Machine":          ", ".join(fixed_ms) if fixed_ms else "—",
            "Monthly_Indent":         round(monthly, 0),
            "Daily_Indent":           round(daily, 2),
            "Safety_Ceiling_Qty_3d":  target_qty,
            "Inv_Before":             round(inv_b, 0),
            "Days_Coverage_Before":   days_before,
            "Produced_Today":         produced,
            "Inv_After":              inv_after,
            "Days_Coverage_After":    days_after,
            "Gap_to_Ceiling_Qty":     max(0, gap_qty),
            "Gap_to_Ceiling_Days":    gap_days,
            "Buffer_Status":          status,
            "OPD_Cap_Today_Days":     cap,
            "Max_Producible_Qty":     max_prod,
            "Est_Days_to_Ceiling":    est_days,
            "Required_Terminals":     terminals_for_display(p),
            "Terminal_Block_Reason":  t_reason if t_blocked else "",
            "Scheduled_Today":        ("YES" if produced > 0
                                       else "SKIPPED — AT CEILING" if inv_b >= target_qty
                                       else "NO — no capacity"),
            "Skip_Reason":            skip_reason if skip else "",
        })
    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "AT_CEILING": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort", "Gap_to_Ceiling_Days"], ascending=[True, False])
        df = df.drop(columns=["_sort"]).reset_index(drop=True)
    return df


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0
        days_cov   = inv / daily if daily > 0 else 0
        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip and inv >= SAFETY_DAYS * daily:
            status = "AT 3-DAY CEILING — SKIP"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"
        fixed_ms = get_fixed_machines_for_part(p)
        rows.append({
            "Part":             p,
            "Color":            part_color.get(p, "UNKNOWN"),
            "Category":         part_category.get(p, "Stranger"),
            "Fixed_Machine":    ", ".join(fixed_ms) if fixed_ms else "—",
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Days_Coverage":    round(days_cov, 2),
            "Safety_Ceiling":   SAFETY_DAYS,
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Required_Terminals": terminals_for_display(p),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)


SPECIALIZED_MACHINE_THRESHOLD = 3

def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        machine_parts[m] = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
    specialized_machines = {m for m, mp in machine_parts.items() if 0 < len(mp) <= SPECIALIZED_MACHINE_THRESHOLD}
    return specialized_machines, {m: machine_parts[m] for m in specialized_machines}, machine_parts

# =============================================================
# MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    global priority_scores_global
    _reset_terminal_running_stock()

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"  Safety ceiling: {SAFETY_DAYS} days")
    print(f"  OPD cap: {OPD_SCENARIO_3}× daily  |  Util: {UTIL_TARGET_PCT:.0f}%")
    print(f"  Machine order: Runner >>>> Repeater >>>> Stranger (enforced everywhere)")
    print(f"  Terminal logic: Runner=85%+minRun | Rep/Stranger=fullDailyIndent")
    print(f"  Fixed machines: {len(vt_fixed_map)} machine(s) with reserved assignments")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  Effective OPD cap today: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    priority_scores_global = priority_scores

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []

    specialized_machines, spec_machine_parts, _ = detect_specialized_machines(list(parts))

    # ── Print fixed machine status ────────────────────────────────
    if vt_fixed_map:
        print(f"\n  FIXED MACHINE STATUS:")
        print(f"  {'Machine':<25} {'Fixed Part':<30} {'Inv':>8} {'Days':>6}  {'Status'}")
        print(f"  {'─'*90}")
        for fm, fps in sorted(vt_fixed_map.items()):
            for fp in fps:
                fp_inv   = current_inventory.get(fp, 0.0)
                fp_daily = indent_daily.get(fp, 0.0)
                fp_days  = fp_inv / fp_daily if fp_daily > 0 else 0
                is_open  = fixed_machine_is_open(fm, current_inventory)
                status   = f"OPEN (all fixed parts >= {SAFETY_DAYS}d inv)" if is_open else \
                           f"RESERVED ({fp_days:.1f}d inv < {SAFETY_DAYS}d floor)"
                print(f"  {fm:<25} {fp:<30} {fp_inv:>8.0f} {fp_days:>6.1f}  [{status}]")

    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts):")
    if specialized_machines:
        for m in sorted(specialized_machines, key=lambda m: machine_part_count.get(m, 99)):
            mparts = spec_machine_parts.get(m, [])
            cnt    = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} parts: {', '.join(mparts)}")
    else:
        print(f"    None")

    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0
        and part_category.get(p, "Stranger") in ("Runner", "Repeater")
        and vt_compat.get(p)
    ]

    if zero_inv_rr:
        print(f"\n  DISPLACEMENT PRE-PASS  ({len(zero_inv_rr)} zero-inv Runner/Repeater parts)")
    else:
        print(f"\n  DISPLACEMENT PRE-PASS  — no zero-inv R/R parts  ✓")

    # TWO-TIER SORT: Runner > Repeater > Stranger, within each by priority score
    sorted_active = sorted(
        active_parts,
        key=lambda p: (
            CATEGORY_TIER.get(part_category.get(p, "Stranger"), 2),
            -priority_scores.get(p, 0)
        )
    )

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  ORDER: All Runners first → All Repeaters → All Strangers")
    print(f"  {'Part':<30} {'Cat':<10} {'Color':<10} {'Score':>6} {'Days':>5} "
          f"{'Status':<15} {'Machine(s)':<25} {'Run':>5} {'Qty':>8} {'Term':>12}")
    print(f"  {'─'*130}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        color    = part_color.get(part, "UNKNOWN")
        days_cov = inv_now / daily if daily > 0 else 999

        buf_label = ("CRITICAL"     if inv_now == 0 else
                     "BELOW_SAFETY" if days_cov < SAFETY_DAYS else
                     "AT_CEILING")

        # Terminal status for display
        t_allowed, t_reason_disp, t_qty_disp = terminal_availability_check(
            part, category, use_running_stock=True)
        term_disp = f"T:{t_qty_disp:.0f}pcs" if t_qty_disp != float("inf") else "T:OK"
        if not t_allowed:
            term_disp = "T:BLOCKED"

        # Soft Runner deferral
        if (category == "Runner"
                and 1.0 <= days_cov < SAFETY_DAYS
                and runner_should_soft_defer(part, machine_hours)):
            deferred.append({
                "Part": part, "Color": color, "Category": category,
                "Reason": (f"Soft Runner defer — inv {days_cov:.1f}d (1-3d band) "
                           f"and ALL compatible machines >{RUNNER_DEFER_MACHINE_UTIL_PCT}% util"),
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{'SOFT-DEFERRED':<15}  all machines busy")
            continue

        if monthly == 0:
            deferred.append({"Part": part, "Color": color, "Category": category,
                              "Reason": "Monthly indent = 0"})
            continue

        # Terminal gate check (already done in should_skip, but re-check with running stock)
        if not t_allowed:
            not_planned.append({
                "Part":                part,
                "Color":               color,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Fixed_Machine":       ", ".join(get_fixed_machines_for_part(part)) or "—",
                "Reason":              t_reason_disp,
                "Action_Needed":       "Restore terminal(s) / check stock levels",
                "Terminal_Required":   ", ".join(part_terminals.get(part, [])),
                "Terminal_Available_Qty": f"{t_qty_disp:.0f}" if t_qty_disp != float("inf") else "N/A",
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ TERMINAL BLOCKED  {t_reason_disp[:40]}")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Color": color, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Fixed_Machine": ", ".join(get_fixed_machines_for_part(part)) or "—",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(part, scenario_id, machine_hours, machine_last_part,
                               current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            purges = sum(1 for r in new_rows if r.get("Color_Purge") == "Yes")
            flag  += f" [PURGE×{purges}]" if purges > 0 else ""
            # Flag if fixed machine was used
            fixed_used = sum(1 for r in new_rows if r.get("Fixed_Machine_Used") == "YES")
            flag += f" [FIXED×{fixed_used}]" if fixed_used > 0 else ""
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  {machines_str:<25}  {total_run:>5.2f}  {total_qty:>8.0f}  {term_disp:>12}  ✓{flag}")
        else:
            not_planned.append({
                "Part":                part,
                "Color":               color,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Fixed_Machine":       ", ".join(get_fixed_machines_for_part(part)) or "—",
                "Reason":              "No compatible machine has capacity (fixed machine reserved or all machines full)",
                "Action_Needed":       "Review matrix, fixed machine assignments, or add machines",
                "Terminal_Required":   ", ".join(part_terminals.get(part, [])) or "—",
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ NO CAPACITY")

    # Post-primary displacement for zero-inv Runner/Repeater
    unscheduled_zero_rr = [p for p in zero_inv_rr if p not in already_planned]
    if unscheduled_zero_rr:
        print(f"\n  DISPLACEMENT PASS  ({len(unscheduled_zero_rr)} zero-inv R/R unscheduled)")
        displaced_log = []
        for part in unscheduled_zero_rr:
            success = displace_for_zero_inv(
                part, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores)
            if success:
                displaced_log.append(part)
                not_planned[:] = [r for r in not_planned if r.get("Part") != part]
            else:
                print(f"      ↳ DISPLACEMENT FAILED  {part}  — no suitable victim")
    else:
        displaced_log = []
        print(f"\n  DISPLACEMENT PASS  — no unscheduled zero-inv R/R  ✓")

    # Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned, current_inventory, scenario_id, priority_scores)

    # CO stagger
    print(f"\n  CO Quantity Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min after CO)")
    n_adj = stagger_co_by_quantity(plan, vt_machines, scenario_id, current_inventory)
    print(f"    {'No adjustments needed  ✓' if n_adj == 0 else str(n_adj) + ' adjustment(s) made'}")

    stagger_changeovers(plan, vt_machines)

    # Build views
    multi_machine_df  = build_multi_machine_view(plan)
    prod_vs_indent_df = build_production_vs_indent(plan, list(parts))
    inv_target_df     = build_inventory_target_sheet(plan, list(parts), scenario_id)

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        fixed_ms = get_fixed_machines_for_part(p)
        indent_status_rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Fixed_Machine":      ", ".join(fixed_ms) if fixed_ms else "—",
            "Fixed_Machine_Used": row.get("Fixed_Machine_Used", "No"),
            "Machine":            row.get("Machine", "—"),
            "Color_Purge":        row.get("Color_Purge", "No"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Buffer_Status":      (
                "CRITICAL"     if inv_b == 0 else
                "BELOW_SAFETY" if (daily > 0 and inv_b / daily < SAFETY_DAYS) else
                "AT_CEILING"
            ),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
            "Required_Terminals": terminals_for_display(p),
            "Terminal_Criticality": row.get("Terminal_Criticality", "—"),
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"], ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        fixed_ms = get_fixed_machines_for_part(p)
        inv_rows.append({
            "Part":            p,
            "Color":           part_color.get(p, "UNKNOWN"),
            "Category":        part_category.get(p, "Stranger"),
            "Fixed_Machine":   ", ".join(fixed_ms) if fixed_ms else "—",
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Safety_Ceiling":  SAFETY_DAYS,
            "Status":          (
                "AT_CEILING" if days_cov >= SAFETY_DAYS else
                "OK"         if days_cov >= 1 else
                "CRITICAL"
            ),
            "Required_Terminals": terminals_for_display(p),
        })

    # Machine utilisation
    mach_rows = []
    for m in vt_machines:
        used        = machine_hours.get(m, 0)
        parts_run   = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count    = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        purge_count = sum(1 for r in plan if r["Machine"] == m and r.get("Color_Purge") == "Yes")
        colors_on_machine = list({part_color.get(r["Part"], "UNKNOWN")
                                   for r in plan if r["Machine"] == m})
        util_pct    = round(used / AVAILABLE_HOURS * 100, 1)
        runner_count   = sum(1 for p in parts_run if part_category.get(p) == "Runner")
        repeater_count = sum(1 for p in parts_run if part_category.get(p) == "Repeater")
        stranger_count = sum(1 for p in parts_run if part_category.get(p) == "Stranger")
        # Fixed machine info
        is_fixed_m    = is_fixed_machine(m)
        fixed_parts_m = vt_fixed_map.get(m, [])
        non_fixed_on_m = [p for p in parts_run if p not in fixed_parts_m]
        mach_rows.append({
            "Machine":              m,
            "Is_Fixed_Machine":     "YES" if is_fixed_m else "No",
            "Fixed_Parts":          ", ".join(fixed_parts_m) if fixed_parts_m else "—",
            "Non_Fixed_Parts_Run":  ", ".join(non_fixed_on_m) if non_fixed_on_m else "—",
            "Colors_Today":         ", ".join(sorted(colors_on_machine)),
            "Color_Purges":         purge_count,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "OK"       if used >= AVAILABLE_HOURS * 0.90 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Runners_On_Machine":   runner_count,
            "Repeaters_On_Machine": repeater_count,
            "Strangers_On_Machine": stranger_count,
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df  = pd.DataFrame(micro_idle)   if micro_idle   else pd.DataFrame()
    plan_df   = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df    = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df    = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    mach_df   = pd.DataFrame(mach_rows)
    inv_df    = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date",   str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",        scenario_desc)
        plan_df.insert(2, "Working_Days",    WORKING_DAYS)
        plan_df.insert(3, "Safety_Ceiling",  SAFETY_DAYS)
        plan_df.insert(4, "OPD_Cap_Today",   opd_cap(scenario_id))

    n_displaced = len(displaced_log)
    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned           : {len(already_planned)}")
    print(f"    Displaced (zero-inv R/R): {n_displaced}")
    print(f"    Not planned             : {len(not_planned)}")
    print(f"    Deferred                : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization         : {mach_df['Utilization_%'].mean():.1f}%")
        total_purges = mach_df["Color_Purges"].sum()
        print(f"    Total colour purges     : {total_purges}")
    if not inv_target_df.empty:
        at_c = (inv_target_df["Buffer_Status"] == "AT_CEILING").sum()
        bls  = (inv_target_df["Buffer_Status"] == "BELOW_SAFETY").sum()
        crt  = (inv_target_df["Buffer_Status"] == "CRITICAL").sum()
        print(f"\n    Inventory target status (after today):")
        print(f"      AT_CEILING  (≥3 days) : {at_c}")
        print(f"      BELOW_SAFETY (<3 days): {bls}")
        print(f"      CRITICAL  (0 pcs)     : {crt}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df,
            inv_target_df)

# =============================================================
# PART AUDIT
# =============================================================

vt_parts = data_valid[data_valid["Material"].isin(vt_matrix["Part"])]["Material"].unique()
all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv      = inventory.get(part, 0.0)
    r_val    = rate.get(part, None)
    monthly  = indent_monthly.get(part, 0.0)
    daily    = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)
    color    = part_color.get(part, "UNKNOWN")
    days_cov = inv / daily if daily > 0 else 0
    part_terms    = part_terminals.get(part, [])
    t_blk, t_rsn  = is_terminal_blocked(part)
    terms_str     = ", ".join(part_terms) if part_terms else "—"
    terms_blocked = blocked_terminals_for_display(part)
    fixed_ms      = get_fixed_machines_for_part(part)
    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif daily <= MIN_DAILY_INDENT:
        status, gate, reason = "SKIPPED (LOW INDENT)", "GATE 4a", f"Daily ≤ {MIN_DAILY_INDENT}"
    elif (monthly / r_val if r_val else 0) <= MIN_INDENT_HOURS:
        status, gate, reason = "SKIPPED (TRIVIAL RUN)", "GATE 4b", f"Monthly hrs ≤ {MIN_INDENT_HOURS}h"
    elif t_blk:
        status, gate, reason = "TERMINAL BLOCKED", "GATE 4c", t_rsn
    elif daily > 0 and inv >= SAFETY_DAYS * daily:
        status, gate, reason = f"AT {SAFETY_DAYS}-DAY CEILING — SKIP TODAY", "GATE 5", \
            f"Inv ({inv:.0f}) ≥ {SAFETY_DAYS}×daily ({SAFETY_DAYS*daily:.0f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"
    audit_rows.append({
        "Part":              part,
        "Color":             color,
        "Category":          part_category.get(part, "Stranger"),
        "Fixed_Machine":     ", ".join(fixed_ms) if fixed_ms else "—",
        "Gate_Failed":       gate,
        "Reason":            reason,
        "Terminals_Required":terms_str,
        "Terminals_Down":    terms_blocked,
        "Terminal_Block_Detail": t_rsn if t_blk else "",
        "Monthly_Indent":    round(monthly, 0),
        "Daily_Indent":      round(daily, 2),
        "Inventory":         round(inv, 0),
        "Days_Coverage":     round(days_cov, 2),
        "Safety_Ceiling":    SAFETY_DAYS,
        "Tools":             tools,
        "Rate_Per_Hour":     round(r_val, 2) if r_val else "—",
        "Cycle_Time":        ct_raw,
        "Cavity":            cv_raw,
        "Status":            status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<50}: {count:>4}")

# =============================================================
# TERMINAL STATUS SHEET BUILDER
# =============================================================

def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_stock.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)
    for t in sorted(all_terminals_known):
        stock  = terminal_stock.get(t, float("inf"))
        avail  = stock > 0
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_blocked = []
        for p in parts_needing:
            _, t_reason, _ = terminal_availability_check(p, part_category.get(p, "Stranger"))
            if not avail or (terminal_stock.get(t, 0) < (RUNNER_TERMINAL_MIN_PCT/100.0) * indent_daily.get(p, 0)
                             and part_category.get(p, "Stranger") == "Runner"):
                parts_blocked.append(p)
        rows.append({
            "Terminal":              t,
            "Current_Stock":         round(stock, 0) if stock != float("inf") else "∞",
            "Available_Today":       "YES" if avail else "NO — DOWN",
            "Parts_Requiring_Count": len(parts_needing),
            "Parts_Requiring":       ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Runner_Parts_Needing":  ", ".join(p for p in parts_needing if part_category.get(p) == "Runner") or "—",
            "85pct_Runner_Threshold_Met": (
                "N/A" if not any(part_category.get(p) == "Runner" for p in parts_needing) else
                "YES" if all(stock >= (RUNNER_TERMINAL_MIN_PCT/100.0) * indent_daily.get(p, 0)
                             for p in parts_needing if part_category.get(p) == "Runner") else "NO"
            ),
            "Parts_Blocked_Count":   len(parts_blocked),
            "Parts_Blocked":         ", ".join(sorted(parts_blocked)) if parts_blocked else "—",
            "Impact":                (
                "BLOCKING — reschedule those parts" if not avail and parts_needing
                else "RUNNER BELOW 85% THRESHOLD" if any(
                    stock < (RUNNER_TERMINAL_MIN_PCT/100.0) * indent_daily.get(p, 0)
                    for p in parts_needing if part_category.get(p) == "Runner")
                else "No impact today" if parts_needing
                else "No parts use this terminal"
            ),
        })
    df = pd.DataFrame(rows)
    return df, []

vt_terminal_status_df, parts_no_terminal = build_terminal_status_sheet()

if not vt_terminal_status_df.empty:
    down_rows = vt_terminal_status_df[vt_terminal_status_df["Available_Today"] == "NO — DOWN"]
    total_blocked_today = down_rows["Parts_Blocked_Count"].sum()
    print(f"\n  Terminal Status Summary:")
    print(f"    Terminals DOWN today     : {(vt_terminal_status_df['Available_Today']=='NO — DOWN').sum()}")
    print(f"    Parts blocked today      : {int(total_blocked_today)}")
    runner_below_85 = (vt_terminal_status_df["85pct_Runner_Threshold_Met"] == "NO").sum()
    print(f"    Terminals below 85% (R)  : {runner_below_85}")

# =============================================================
# RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent, vt_inv_target) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()
    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default
    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        is_fixed_m    = is_fixed_machine(m)
        fixed_parts_m = vt_fixed_map.get(m, [])
        for _, pr in machine_rows.iterrows():
            p    = pr.get("Part", "—")
            co_h = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":            m,
                "Is_Fixed_Machine":   "YES" if is_fixed_m else "No",
                "Fixed_Parts":        ", ".join(fixed_parts_m) if fixed_parts_m else "—",
                "Part":               p,
                "Color":              part_color.get(p, "UNKNOWN"),
                "Category":           part_category.get(p, "Stranger"),
                "Fixed_Machine_Used": pr.get("Fixed_Machine_Used", "No"),
                "Role":               pr.get("Role", "Primary"),
                "Tools_Available":    pr.get("Tools_Available", 1),
                "Priority_Score":     round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":      round(r_val, 2),
                "Run_Hours":          round(run_h, 2),
                "Changeover_Hrs":     round(co_h, 3),
                "Changeover_Needed":  pr.get("Changeover", "No") or "No",
                "Color_Purge":        pr.get("Color_Purge", "No") or "No",
                "Production_Qty":     round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":       round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":       round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":     round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Required_Terminals": pr.get("Required_Terminals", "—"),
                "Terminal_Criticality": pr.get("Terminal_Criticality", "—"),
                "Type":               pr.get("Type", "Primary") or "Primary",
                "Row_Type":           "Part",
            })
        co_total    = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total   = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total   = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count    = int(machine_rows["Changeover"].eq("Yes").sum())
        purge_count = int(machine_rows["Color_Purge"].eq("Yes").sum()) if "Color_Purge" in machine_rows.columns else 0
        hrs_total   = round(co_total + run_total, 2)
        colors_list = sorted({part_color.get(p, "UNKNOWN") for p in machine_rows["Part"]})
        rows.append({
            "Machine":            m,
            "Is_Fixed_Machine":   "YES" if is_fixed_m else "No",
            "Fixed_Parts":        ", ".join(fixed_parts_m) if fixed_parts_m else "—",
            "Part":               f"TOTAL — {m}",
            "Color":              ", ".join(colors_list),
            "Category":           "—",
            "Fixed_Machine_Used": "—",
            "Role":               "—",
            "Tools_Available":    "—",
            "Priority_Score":     "—",
            "Rate_Per_Hour":      "—",
            "Run_Hours":          round(run_total, 2),
            "Changeover_Hrs":     round(co_total, 2),
            "Changeover_Needed":  f"{co_count} changeover(s)",
            "Color_Purge":        f"{purge_count} purge(s)",
            "Production_Qty":     round(qty_total, 0),
            "Daily_Indent":       "—",
            "Today_Target":       "—",
            "Monthly_Indent":     "—",
            "Required_Terminals": "—",
            "Terminal_Criticality": "—",
            "Type":               (f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  "
                                   f"Idle {round(AVAILABLE_HOURS - hrs_total, 2)}h  |  "
                                   f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"),
            "Row_Type":           "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0
    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_min      = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h
        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = (before_color != after_color
                        and before_color != "UNKNOWN" and after_color != "UNKNOWN")
        note = ""
        if wait_min > 0:
            note = f"Machine may need to wait ~{wait_min}min."
        if color_change:
            note = (note + "  " if note else "") + f"COLOUR CHANGE: {before_color} → {after_color}. 10-min purge required."
        rows.append({
            "Queue_Position":   pos,
            "Machine":          ev["machine"],
            "Part_Before":      ev["part_before"],
            "Color_Before":     before_color,
            "Part_After":       ev["part_after"],
            "Color_After":      after_color,
            "Color_Change":     "YES — PURGE" if color_change else "No",
            "CO_Duration_Min":  round(co_h * 60, 1),
            "Note":             note if note else "Tool changer available immediately.",
        })
    return pd.DataFrame(rows)

# =============================================================
# EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_CO_Queue":             "375623",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Inventory_Target":     "1B4F72",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
    "VT_Terminal_Status":      "7B1C1C",
}

STATUS_FILLS = {
    "FULL":         PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":         PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":      PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":    PatternFill("solid", fgColor="FFC7CE"),
    "OK":           PatternFill("solid", fgColor="C6EFCE"),
    "LOW":          PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":     PatternFill("solid", fgColor="FFC7CE"),
    "AT_CEILING":   PatternFill("solid", fgColor="C6EFCE"),
    "BELOW_SAFETY": PatternFill("solid", fgColor="FFEB9C"),
    "YES ✓":        PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":         PatternFill("solid", fgColor="FFC7CE"),
    "OVER":         PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":        PatternFill("solid", fgColor="FFC7CE"),
    "MET":          PatternFill("solid", fgColor="C6EFCE"),
    "YES — PURGE":  PatternFill("solid", fgColor="FFC7CE"),
    "Yes":          PatternFill("solid", fgColor="FFEB9C"),
    "YES":          PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":                   PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":                   PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":                      PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                             PatternFill("solid", fgColor="EDEDED"),
    f"AT {SAFETY_DAYS}-DAY CEILING — SKIP TODAY": PatternFill("solid", fgColor="C6EFCE"),
    "AT 3-DAY CEILING — SKIP":             PatternFill("solid", fgColor="C6EFCE"),
    "ZERO/MISSING CYCLE TIME":             PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                    PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":                 PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                    PatternFill("solid", fgColor="C6EFCE"),
    "NO — DOWN":                           PatternFill("solid", fgColor="FFC7CE"),
    "BLOCKING — reschedule those parts":   PatternFill("solid", fgColor="FFC7CE"),
    "RUNNER BELOW 85% THRESHOLD":          PatternFill("solid", fgColor="FFEB9C"),
    "No impact today":                     PatternFill("solid", fgColor="C6EFCE"),
    "TERMINAL BLOCKED":    PatternFill("solid", fgColor="FFC7CE"),
    "HIGHLY CRITICAL":     PatternFill("solid", fgColor="FF4500"),
    "OPERATIONAL":         PatternFill("solid", fgColor="C6EFCE"),
    "DOWN — BLOCKING":     PatternFill("solid", fgColor="FF0000"),
    "NO":                  PatternFill("solid", fgColor="FFEB9C"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction", "Buffer_Status",
                             "Scheduled_Today", "Color_Change", "Color_Purge",
                             "Available_Today", "Impact", "Terminal_Status",
                             "85pct_Runner_Threshold_Met", "Fixed_Machine_Used",
                             "Is_Fixed_Machine"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_inv_target_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Inventory_Target"])
    headers    = [c.value for c in ws[1]]
    status_col = headers.index("Buffer_Status") + 1 if "Buffer_Status" in headers else None
    row_fills  = {
        "CRITICAL":     PatternFill("solid", fgColor="FFD7D7"),
        "BELOW_SAFETY": PatternFill("solid", fgColor="FFF2CC"),
        "AT_CEILING":   PatternFill("solid", fgColor="E2EFDA"),
    }
    for row in ws.iter_rows(min_row=2):
        if not status_col:
            continue
        status = str(row[status_col - 1].value)
        fill   = row_fills.get(status)
        if fill:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000", "FFFFFFFF"):
                    cell.fill = fill
        if status == "CRITICAL":
            for cell in row:
                cell.font = Font(bold=True)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"), PatternFill("solid", fgColor="F0FDF4")]
    fixed_fill   = PatternFill("solid", fgColor="FFF3CD")   # light amber for fixed machines
    co_fill      = PatternFill("solid", fgColor="FEF9C3")
    purge_fill   = PatternFill("solid", fgColor="FFC7CE")
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30
    headers           = [cell.value for cell in ws[1]]
    row_type_col      = headers.index("Row_Type")          + 1 if "Row_Type"          in headers else None
    co_col            = headers.index("Changeover_Needed") + 1 if "Changeover_Needed" in headers else None
    purge_col         = headers.index("Color_Purge")       + 1 if "Color_Purge"       in headers else None
    machine_col       = headers.index("Machine")           + 1 if "Machine"           in headers else None
    is_fixed_col      = headers.index("Is_Fixed_Machine")  + 1 if "Is_Fixed_Machine"  in headers else None
    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type  = row[row_type_col-1].value if row_type_col else ""
        machine   = row[machine_col-1].value  if machine_col  else ""
        is_fixed  = str(row[is_fixed_col-1].value) if is_fixed_col else "No"
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            base_fill = fixed_fill if is_fixed == "YES" else part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = base_fill
                cell.alignment = Alignment(vertical="center")
            if co_col and str(row[co_col-1].value) == "Yes":
                row[co_col-1].fill = co_fill
            if purge_col and str(row[purge_col-1].value) == "Yes":
                row[purge_col-1].fill = purge_fill
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])
    headers  = [c.value for c in ws[1]]
    gap_col  = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    over_fill  = PatternFill("solid", fgColor="DDEBF7")
    under_fill = PatternFill("solid", fgColor="FFC7CE")
    met_fill   = PatternFill("solid", fgColor="C6EFCE")
    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])
    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None
    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")
    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


def style_co_queue_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_CO_Queue"])
    headers    = [c.value for c in ws[1]]
    cc_col     = headers.index("Color_Change") + 1 if "Color_Change" in headers else None
    purge_fill = PatternFill("solid", fgColor="FFC7CE")
    for row in ws.iter_rows(min_row=2):
        if cc_col and str(row[cc_col-1].value).startswith("YES"):
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000", "FFFFFFFF"):
                    cell.fill = purge_fill
            row[cc_col-1].font = Font(bold=True, color="7B1C1C")


print(f"\nWriting output → {output_path}")

vt_mw   = build_machine_wise_plan(vt_plan)
vt_co_q = build_co_queue(
    [{k:v for k,v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines
)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Inventory_Target":     vt_inv_target,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
    "VT_Terminal_Status":      vt_terminal_status_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine"      in wb.sheetnames: style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames: style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts"  in wb.sheetnames: style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])
if "VT_Inventory_Target"     in wb.sheetnames: style_inv_target_sheet(wb["VT_Inventory_Target"])
if "VT_CO_Queue"             in wb.sheetnames: style_co_queue_sheet(wb["VT_CO_Queue"])
if "VT_Terminal_Status"      in wb.sheetnames: style_sheet(wb["VT_Terminal_Status"], HEADER_COLORS["VT_Terminal_Status"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent",
            "VT_Multi_Machine_Parts", "VT_Inventory_Target",
            "VT_CO_Queue", "VT_Terminal_Status"):
        style_sheet(wb[sheet_name], header_hex)

if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = headers_is.index("Meets_Daily_Indent") + 1 if "Meets_Daily_Indent" in headers_is else None
    covers_col = headers_is.index("Covers_With_Inv")    + 1 if "Covers_With_Inv"    in headers_is else None
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V10 Complete  —  {PLANNING_DATE}")
print(f"  Safety ceiling: {SAFETY_DAYS} days  |  OPD cap: {OPD_SCENARIO_3}× daily")
print(f"  Colour purge penalty : {int(COLOR_PURGE_HRS*60)} min")
print(f"  Priority: Runner >>>> Repeater >>>> Stranger (enforced everywhere)")
print(f"  Terminal rules:")
print(f"    Runner   : ALL terminals >= {RUNNER_TERMINAL_MIN_PCT:.0f}% daily indent + min {MIN_RUN_HOURS}h run")
print(f"    Repeater : terminals must cover FULL daily indent")
print(f"    Stranger : terminals must cover FULL daily indent")
print(f"  Fixed machine rules:")
print(f"    Reserved : fixed part inv < {SAFETY_DAYS}d → machine for that part ONLY")
print(f"    Open     : ALL fixed parts inv >= {SAFETY_DAYS}d → any compatible part allowed")
print(f"    Preference: fixed parts always try their designated machine first")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<52}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_inv_target.empty:
    at_c = (vt_inv_target["Buffer_Status"] == "AT_CEILING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory status (end of day):")
    print(f"    AT_CEILING  (≥3 days) : {at_c:>4}  — skip tomorrow")
    print(f"    BELOW_SAFETY (<3 days): {bls:>4}  — continue building")
    print(f"    CRITICAL  (0 pcs)     : {crt:>4}  — displacement eligible tomorrow")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average        : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED      : {(vt_mach['Status']=='UNDERUSED').sum()} machines")
    print(f"    Total purges   : {vt_mach['Color_Purges'].sum()} colour changeovers")
    fixed_machines_used = (vt_mach['Is_Fixed_Machine'] == 'YES').sum()
    print(f"    Fixed machines : {fixed_machines_used}")

if not vt_terminal_status_df.empty:
    down_today    = (vt_terminal_status_df["Available_Today"] == "NO — DOWN").sum()
    blocked_today = int(vt_terminal_status_df["Parts_Blocked_Count"].sum())
    print(f"\n  Terminal constraint:")
    print(f"    Terminals DOWN today     : {down_today}")
    print(f"    Parts blocked            : {blocked_today}")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"  UPDATE DAILY: PLANNING_DATE = date(2026, 4, 8)")
print(f"{'='*65}")